# Mango

Mira qué tipos de cliente hay en el centro comercial, y a quién se parece alguien nuevo, antes de mandarle una campaña.

Son 200 personas del Mall Customers. El grupo lo deciden el ingreso anual y las ganas de comprar. La edad y el género ayudan a contar quién hay dentro de cada grupo, pero no parten los grupos.


## Primero, mirar a la gente

Abajo están las primeras filas y un resumen. El ingreso está en miles de dólares. Las ganas de comprar van de 1 a 100: 100 es quien más compra en la tienda.

![Cómo se reparte la edad, el ingreso y las ganas de comprar](../report/figures/boxplots.png)

In [1]:
from mango.data.quality import load_customers, screen_customers, summarize_customers

raw = load_customers()
ready, quality = screen_customers(raw)
summary = summarize_customers(ready)
raw.head()


,customer_id,gender,age,annual_income_k,spending_score
0,1,Male,19,15,39
1,2,Male,21,15,81
2,3,Female,20,16,6
3,4,Female,23,16,77
4,5,Female,31,17,40


In [2]:
summary["numeric_summary"]


{'age': {'count': 200,
  'mean': 38.85,
  'std': 13.96900733155888,
  'min': 18.0,
  'q1': 28.75,
  'median': 36.0,
  'q3': 49.0,
  'max': 70.0},
 'annual_income_k': {'count': 200,
  'mean': 60.56,
  'std': 26.264721165271244,
  'min': 15.0,
  'q1': 41.5,
  'median': 61.5,
  'q3': 78.0,
  'max': 137.0},
 'spending_score': {'count': 200,
  'mean': 50.2,
  'std': 25.823521668370173,
  'min': 1.0,
  'q1': 34.75,
  'median': 50.0,
  'q3': 73.0,
  'max': 99.0}}

In [3]:
summary["gender_counts"]


{'Female': 112, 'Male': 88}

### ¿Qué variables son más adecuadas para segmentar clientes?

Las que sirven para decidir una campaña: cuánto puede gastar una persona y cuánto gasta de verdad. Eso es el ingreso anual y las ganas de comprar.

El código de cliente no dice nada de su comportamiento. El género no entra en la cuenta, porque K-Means mide distancias y el género no es una escala. Lo miramos después, cuando ya sabemos el grupo. La edad también se cuenta dentro de cada grupo. Si la usamos para partir, los grupos quedan más revueltos. Eso se comprueba más abajo.


In [4]:
quality["iqr_fences"]


{'age': {'lower': -1.625, 'upper': 79.375, 'count': 0, 'customer_ids': []},
 'annual_income_k': {'lower': -13.25,
  'upper': 132.75,
  'count': 2,
  'customer_ids': [199, 200]},
 'spending_score': {'lower': -22.625,
  'upper': 130.375,
  'count': 0,
  'customer_ids': []}}

In [5]:
{
    "modified_z": quality["modified_z"],
    "mahalanobis_ids": quality["mahalanobis"]["customer_ids"],
    "rows_removed": quality["rows_removed"],
    "statistical_tail_count": quality["statistical_tail_count"],
}


{'modified_z': {'age': {'threshold': 3.5, 'count': 0, 'customer_ids': []},
  'annual_income_k': {'threshold': 3.5, 'count': 0, 'customer_ids': []},
  'spending_score': {'threshold': 3.5, 'count': 0, 'customer_ids': []}},
 'mahalanobis_ids': [3, 199, 200],
 'rows_removed': 0,
 'statistical_tail_count': 3}

### ¿Se identifican valores atípicos?

Hay extremos, y se quedan.

No falta ningún dato, no hay personas repetidas y nadie tiene una edad imposible, un ingreso negativo o unas ganas de comprar fuera de 1 a 100. No se eliminó a nadie.

La regla de Tukey (1,5 veces el rango intercuartílico) marca solo a dos personas por el ingreso: la 199 y la 200. Las dos ganan 137 mil dólares al año, justo por encima del corte de 132,75. La z modificada, con umbral 3,5, no marca a nadie.

Mahalanobis, mirando edad, ingreso y ganas de comprar juntas, señala además a la persona 3. Esa prueba imagina una sola nube. Aquí hay varios grupos, así que lo que marca son las esquinas de grupos reales, no un error al anotar.

La 199 y la 200 son clientes de ingreso alto. Borrarlos sería esconder justo a quien más le importa a una campaña. Se quedan.


In [6]:
summary["correlations"]


{'age': {'age': 1.0,
  'annual_income_k': -0.012398,
  'spending_score': -0.327227},
 'annual_income_k': {'age': -0.012398,
  'annual_income_k': 1.0,
  'spending_score': 0.009903},
 'spending_score': {'age': -0.327227,
  'annual_income_k': 0.009903,
  'spending_score': 1.0}}

### ¿Qué tipo de relaciones se observan entre las variables?

El ingreso y las ganas de comprar casi no se mueven juntos. La correlación es 0,01. No hay una línea: hay grupos. Por eso tiene más sentido partir en grupos que buscar una recta.

La edad y las ganas de comprar bajan un poco juntas (−0,33): a más edad, en general se compra un poco menos, pero la relación es floja. La edad y el ingreso van cada una por su lado (−0,01).

Eso se ve después en el mapa: cinco manchas, no una nube con pendiente.


## Poner el ingreso y las compras en la misma escala

In [7]:
from mango.clustering.model import scale_features
from mango.constants import SEGMENT_FEATURES

scaled, scaler = scale_features(ready, SEGMENT_FEATURES)
list(zip(SEGMENT_FEATURES, scaler.mean_, scaler.scale_))


[('annual_income_k', np.float64(60.56), np.float64(26.19897707926781)),
 ('spending_score', np.float64(50.2), np.float64(25.7588819633151))]

### ¿Por qué es necesario escalar las variables en el algoritmo K-Means?

K-Means junta a cada persona con el centro más cercano. Para medir esa cercanía suma las variables como si fueran la misma cosa. No lo son: el ingreso está en miles de dólares y las ganas de comprar son un puntaje de 1 a 100.

En este archivo las dos se mueven parecido (desviación cerca de 26), así que el efecto es chico. Aun así, el algoritmo se deja llevar por la variable con números más grandes. StandardScaler las deja con media 0 y desviación 1, y así ninguna gana solo por cómo está medida.

Si metiéramos la edad sin escalar, pesaría menos, porque varía menos (cerca de 14). Primero elegimos ingreso y ganas de comprar. Después las escalamos.


## Cuántos grupos conviene usar

![La curva baja rápido y luego se aplana. El doblez está en 5.](../report/figures/elbow.png)

In [8]:
from mango.clustering.model import elbow_curve, geometric_elbow_k
from mango.visualization import plot_elbow

elbow = elbow_curve(scaled)
print("elbow_k", geometric_elbow_k(elbow))
for point in elbow:
    print(point["k"], round(point["inertia"], 1))


elbow_k 5
1 400.0
2 269.7
3 157.7
4 108.9
5 65.6
6 55.1
7 44.9
8 37.2
9 32.4
10 30.0


### ¿Por qué valores cercanos a cero indican mejor ajuste?

Ese número, la inercia, dice qué tan sueltos quedan los puntos alrededor de su centro. Más cerca de cero, más apretados.

Con los datos ya en la misma escala, un solo grupo arranca en 400. Con 5 baja a 65,6. Con 10 sigue en 30.

Cero solo aparece si cada persona es su propio grupo. Ajusta perfecto y no sirve para una campaña. Un número más chico no siempre es mejor.


### ¿Cómo se identifica el punto óptimo de K?

Se busca el codo: el lugar donde la curva deja de caer en picada y se aplana. Además de mirarlo, se calcula el quiebre más marcado entre el primer y el último valor. Cae en 5 grupos.

De 4 a 5 la inercia baja de 108,9 a 65,6. De 5 a 6 solo baja a 55,1. Partir más ya no aporta.


## Qué tan nítido queda cada corte

![Con 5 grupos la separación es la mejor.](../report/figures/silhouette.png)

In [9]:
from mango.clustering.model import silhouette_curve

silhouette = silhouette_curve(scaled)
for point in silhouette:
    print(point["k"], round(point["score"], 4))


2 0.3213
3 0.4666
4 0.4939
5 0.5547
6 0.5399
7 0.5281
8 0.4552
9 0.4571
10 0.4432


### ¿Qué mide el coeficiente de silueta?

Para cada persona mira dos cosas: qué tan cerca está de la gente de su grupo y qué tan cerca está del grupo de al lado. Si está pegada al suyo y lejos del otro, el número sube. El resultado del modelo es el promedio de todas.


### ¿Qué significa un valor cercano a 1?

Los grupos están juntos por dentro y separados entre sí. Aquí el mejor valor es 0,55, con 5 grupos. Se ven claros, sin ser una separación perfecta. En la página ese número es “qué tan claros”.


### ¿Qué implica un valor cercano a 0 o negativo?

Cerca de 0, la persona está en la frontera entre dos grupos. Si el número es negativo, quedó más cerca de otro grupo que del suyo: el corte o las variables no sirven. En este rango lo más flojo es partir en 2, con 0,32. Ningún valor sale negativo.


### ¿Coincide el valor óptimo de K con el método del codo? Justifique.

Sí. El codo y la silueta eligen 5. Con 6 grupos la silueta baja un poco, a 0,54.

También probamos otras parejas. Edad y ganas de comprar llegan a 0,47. Edad e ingreso, a 0,44. Las tres juntas, a 0,43. Ingreso y ganas de comprar, con 5 grupos, llegan a 0,55. Por eso el modelo se queda con esas dos.


## Los cinco tipos de cliente

Cada punto es una persona. A la derecha, quién compra más. Arriba en ingreso, quién puede gastar más. La cruz es el cliente típico del grupo.

![Mapa de los cinco grupos](../report/figures/clusters.png)

In [10]:
from mango.orchestration.pipeline import SegmentationOrchestrator

orchestrator = SegmentationOrchestrator()
analysis = orchestrator.fit()
analysis["selected_k"], round(analysis["model_silhouette"], 4), analysis["k_selection"]["rule"]


(5, 0.5547, 'elbow_and_silhouette_agree')

In [11]:
[
    {
        "profile": cluster["profile"],
        "size": cluster["size"],
        "annual_income_k": round(cluster["centroid"]["annual_income_k"], 1),
        "spending_score": round(cluster["centroid"]["spending_score"], 1),
        "mean_age": round(cluster["mean_age"], 1),
    }
    for cluster in analysis["clusters"]
]


[{'profile': 'high_value',
  'size': 39,
  'annual_income_k': 86.5,
  'spending_score': 82.1,
  'mean_age': 32.7},
 {'profile': 'potential',
  'size': 35,
  'annual_income_k': 88.2,
  'spending_score': 17.1,
  'mean_age': 41.1},
 {'profile': 'average',
  'size': 81,
  'annual_income_k': 55.3,
  'spending_score': 49.5,
  'mean_age': 42.7},
 {'profile': 'budget_constrained',
  'size': 22,
  'annual_income_k': 25.7,
  'spending_score': 79.4,
  'mean_age': 25.3},
 {'profile': 'low_value',
  'size': 23,
  'annual_income_k': 26.3,
  'spending_score': 20.9,
  'mean_age': 45.2}]

### ¿Qué representa cada cluster?

El nombre sale de dónde cae el centro del grupo frente al cliente del medio.

**Clientes de alto valor.** 39 personas. Ingreso cerca de US$ 86.500 al año y compran 82 de 100. Edad media 33. Ya concentran margen y frecuencia.

**Clientes con potencial.** 35 personas. Ingreso cerca de US$ 88.200 y compran solo 17 de 100. Edad media 41. Tienen plata y casi no compran en el centro comercial.

**Clientes promedio.** 81 personas, el grupo más grande. Ingreso cerca de US$ 55.300 y compran 50 de 100. Edad media 43. Están en el centro de la base.

**Clientes de presupuesto ajustado.** 22 personas. Ingreso cerca de US$ 25.700 y compran 79 de 100. Edad media 25. Gastan mucho para lo que ganan.

**Clientes de bajo valor.** 23 personas. Ingreso cerca de US$ 26.300 y compran 21 de 100. Edad media 45. Ingreso bajo y poco gasto.


### ¿Qué diferencias existen entre los grupos obtenidos?

El ingreso parte el mapa en dos: cerca de US$ 26.000 al año, o cerca de US$ 87.000. Dentro de cada lado, las ganas de comprar separan a quien ya compra de quien casi no aparece.

El promedio es el más grande y se queda en el medio de las dos cosas. El de presupuesto ajustado es el más joven. El de bajo valor y el promedio pasan de 42 años.

Ningún grupo es solo de mujeres o solo de hombres. La proporción de mujeres va de 46 % a 61 %. No tiene sentido armar la campaña como si un grupo fuera de un solo género.

Si se quitan las dos personas de ingreso 137, la nitidez pasa de 0,55 a 0,56 y los centros se mueven menos de US$ 2.000. El mapa no cambia. Por eso se quedan.


## Qué hacer con cada grupo

In [12]:
for cluster in analysis["clusters"]:
    print(cluster["profile"])
    print(cluster["strategy"])
    print()
print("growth_opportunity", analysis["growth_opportunity"])


high_value
Fidelización, acceso anticipado y servicio personal. Evitar descuentos agresivos que erosionen el margen.

potential
Campañas de conversión: experiencias, personal shopper y surtido aspiracional. Una visita nueva en este grupo mueve más el ingreso que en el resto.

average
Retención con campañas de temporada y venta cruzada moderada. Medir la respuesta antes de subir la inversión.

budget_constrained
Promociones, financiación y formatos accesibles. El volumen no se convierte solo en margen: cuidar el descuento.

low_value
Comunicación de precio y surtido básico. No concentrar el presupuesto de adquisición en este grupo.

growth_opportunity potential


### ¿Qué estrategias de marketing se podrían aplicar a cada grupo?

**Clientes de alto valor.** Fidelización, acceso anticipado y servicio personal. Evitar descuentos agresivos que se coman el margen.

**Clientes con potencial.** Campañas de conversión: experiencias, personal shopper y surtido aspiracional. Una visita nueva aquí mueve más el ingreso que en el resto.

**Clientes promedio.** Retención con campañas de temporada y venta cruzada moderada. Medir la respuesta antes de subir la inversión.

**Clientes de presupuesto ajustado.** Promociones, financiación y formatos accesibles. Compran, pero el presupuesto limita el ticket. Cuidar el descuento.

**Clientes de bajo valor.** Comunicación de precio y surtido básico. No concentrar ahí el presupuesto de adquisición.


### ¿Qué segmento representa mayor oportunidad de crecimiento?

Clientes con potencial. Son 35 personas, con el ingreso más alto del mapa y unas ganas de comprar de las más bajas. El centro comercial ya los tiene. Una visita nueva de este grupo mueve más plata que empujar a quien ya compra mucho y gana poco.

Los de alto valor se cuidan. Eso es retención, no crecimiento.


## Probarlo

Streamlit y la página muestran este mismo análisis: las figuras, los cinco grupos y un formulario para probar a alguien que todavía no está en la base.

```bash
uv run streamlit run streamlit_app.py
```

La página está en `frontend`:

```bash
pnpm dev
```
